In [ ]:
!apt-get update
!apt-get install -y xvfb ffmpeg
!apt-get install xvfb-run -s
!apt-get install -y xvfb x11-utils
!pip install 'imageio==2.4.0'
!pip install pyvirtualdisplay
!pip install tf-agents
!pip install keras-rl2
!pip install gym
!pip install h5py
!pip install stable-baselines3[extra]

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [108 kB]
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,219 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [1,085 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
0% [Connecting to ppa.launchpadcontent.net (185.125.190.52)]

In [1]:
import os
import io
import time
import base64
import imageio
import IPython
import shutil
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import gym
import random
from gym import Env
from gym.spaces import Discrete, Box
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
import tensorflow as tf
from tensorflow import keras
import tensorflow_hub as hub
import tensorflow_datasets as tfds
from tf_agents.agents.dqn import dqn_agent
from tf_agents.policies import random_tf_policy
from tf_agents.networks import sequential
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.eval import metric_utils
from tf_agents.metrics import tf_metric
from rl.agents.dqn import DQNAgent
from rl.policy import EpsGreedyQPolicy, BoltzmannQPolicy
from rl.memory import SequentialMemory
from google.colab import files
import pyvirtualdisplay
# file_load = files.upload()
# for f in file_load.keys():
#   print('Uploaded file "{Name}" with length {length}.bytes'.format(
#       name =f, length = len(file_load[f])))
print(tf.__version__)

class ctrl_env(Env):
  def __init__(self):
    self.observation_space = Box(low=np.array([0]), high =np.array([100]))
    self.action_space = Discrete(3)
    self.state = 50 +random.randint(-5,5)
    self.episode_length = 50

  def reset(self):
    self.state = 50+random.randint(-5,5)
    self.episode_length = 50
    return self.state

  def step(self,action):
    self.state+= action-1
    self.episode_length-= 1
    if self.state>=47 and self.state<=53:
      reward = 1
    else:
      reward=-1
    if self.episode_length<=0:
      done=True
    else:
      done= False
    #adding noise to make the controller more realistic
    # self.state+= random.randint(-1,1)
    info={}
    return self.state, reward, done, info

  def render():
    pass

2.12.0


In [2]:
env=ctrl_env()
episodes =20
for episode in range(1, episodes+1):
  state=env.reset()
  done=False
  tot_reward=0
  while not done:
    action=env.action_space.sample()
    state,reward,done,info = env.step(action)
    tot_reward+=reward
  print('Episode: {}, Total Reward: {}'.format(episode, tot_reward))

Episode: 1, Total Reward: -50
Episode: 2, Total Reward: -36
Episode: 3, Total Reward: 30
Episode: 4, Total Reward: 18
Episode: 5, Total Reward: 44
Episode: 6, Total Reward: 48
Episode: 7, Total Reward: 32
Episode: 8, Total Reward: 0
Episode: 9, Total Reward: 12
Episode: 10, Total Reward: -16
Episode: 11, Total Reward: -30
Episode: 12, Total Reward: -10
Episode: 13, Total Reward: -46
Episode: 14, Total Reward: 20
Episode: 15, Total Reward: 44
Episode: 16, Total Reward: -32
Episode: 17, Total Reward: -50
Episode: 18, Total Reward: 8
Episode: 19, Total Reward: 20
Episode: 20, Total Reward: 2


/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [3]:
states=env.observation_space.shape
actions =env.action_space.n
def agent_model(states, actions):
  model = tf.keras.models.Sequential()
  model.add(tf.keras.layers.Dense(20, activation='relu',input_shape=states))
  model.add(tf.keras.layers.Dense(10, activation='relu'))
  model.add(tf.keras.layers.Dense(actions))
  return model
model =agent_model(states,actions)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 20)                40        
                                                                 
 dense_1 (Dense)             (None, 10)                210       
                                                                 
 dense_2 (Dense)             (None, 3)                 33        
                                                                 
Total params: 283
Trainable params: 283
Non-trainable params: 0
_________________________________________________________________


In [4]:
def agent(model,actions):
  policy =BoltzmannQPolicy()
  memory = SequentialMemory(limit= 100000, window_length=1)
  dqn =DQNAgent(model=model, policy=policy,memory=memory,nb_actions = actions,
                nb_steps_warmup=5, target_model_update=1e-2)
  return dqn

dqn = agent(model,actions)
dqn.compile(tf.keras.optimizers.legacy.Adam(learning_rate=1e-3), metrics=['mae'])
dqn.fit(env,nb_steps=100000,visualize=False, verbose=1)

Training for 100000 steps ...
Interval 1 (0 steps performed)
    1/10000 [..............................] - ETA: 10:08 - reward: 1.0000

/usr/local/lib/python3.10/dist-packages/keras/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
/usr/local/lib/python3.10/dist-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


10000/10000 [==============================] - 89s 9ms/step - reward: -0.0062
200 episodes - episode_reward: -0.310 [-50.000, 50.000] - loss: 0.461 - mae: 1.942 - mean_q: 1.440

Interval 2 (10000 steps performed)
10000/10000 [==============================] - 89s 9ms/step - reward: 0.1140
200 episodes - episode_reward: 5.700 [-50.000, 50.000] - loss: 0.918 - mae: 4.486 - mean_q: 6.407

Interval 3 (20000 steps performed)
10000/10000 [==============================] - 91s 9ms/step - reward: 0.1072
200 episodes - episode_reward: 5.360 [-50.000, 50.000] - loss: 0.967 - mae: 4.676 - mean_q: 6.657

Interval 4 (30000 steps performed)
10000/10000 [==============================] - 92s 9ms/step - reward: 0.1410
200 episodes - episode_reward: 7.050 [-50.000, 50.000] - loss: 0.879 - mae: 4.441 - mean_q: 6.290

Interval 5 (40000 steps performed)
10000/10000 [==============================] - 97s 10ms/step - reward: 0.1166
200 episodes - episode_reward: 5.830 [-50.000, 50.000] - loss: 0.860 - mae: 

In [7]:
test_rewards= dqn.test(env,nb_episodes=20,visualize=False)
print(np.mean(test_rewards.history['episode_reward']))

Testing for 20 episodes ...
Episode 1: reward: 50.000, steps: 50
Episode 2: reward: -50.000, steps: 50
Episode 3: reward: 50.000, steps: 50
Episode 4: reward: 50.000, steps: 50
Episode 5: reward: 50.000, steps: 50
Episode 6: reward: 50.000, steps: 50
Episode 7: reward: -50.000, steps: 50
Episode 8: reward: 50.000, steps: 50
Episode 9: reward: -50.000, steps: 50
Episode 10: reward: 48.000, steps: 50
Episode 11: reward: -50.000, steps: 50
Episode 12: reward: -50.000, steps: 50
Episode 13: reward: 50.000, steps: 50
Episode 14: reward: -50.000, steps: 50
Episode 15: reward: 50.000, steps: 50
Episode 16: reward: 50.000, steps: 50
Episode 17: reward: -50.000, steps: 50
Episode 18: reward: 48.000, steps: 50
Episode 19: reward: 50.000, steps: 50
Episode 20: reward: 50.000, steps: 50
14.8
